# Monitoring variant: ocudu-zmq/zmq/main.ipynb

Based on the upstream example. The original remains under `examples/ocudu-zmq/main.ipynb`.
Core: **open5gs**; RAN: **ocudu**. Stock Open5GS images are used; legacy custom core KPIs are disabled. Native OCUDU JSON metrics are exported through Telegraf when ran_kpis is enabled.

# **5G Deployment Example**

In this example, we will deploy a 5G network with:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OCUDU gNB with ZeroMQ
- SRSUE with ZeroMQ as simulated UE

Reference: https://docs.ocudu.org/tutorials/srsue/#zeromq-based-setup

First, we install the required dependencies:

## Install Dependencies:

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal serevers:

In [ ]:
%pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
%pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Install the Post-5G orchestrator:

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../../..").resolve()
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade --force-reinstall --no-deps "orchestrator @ git+https://github.com/ifran-project/ifran.git@main"
%pip install --upgrade "git+https://github.com/yassir63/nador.git@main" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


## Input 5G Network Description:

In [ ]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  sim_ru_gnb:
    txGainDB: 0
    rxGainDB: 0

  sim_ru_ue:
    txGainDB: 50
    rxGainDB: 40

ran:
  components:
    gnb:
      type: gnb
      config:
        ru: sim_ru_gnb
        radioConfiguration:
          nrBand: 3
          downlinkArfcn: 368500
          channelBandwidthMhz: 10
          subcarrierSpacingKhz: 15
          ssbPeriodicityMs: 10
          prachConfigurationIndex: 1

ues:
  sim_ue:
    config:
      nrBands: [3]
      ru: sim_ru_ue

slices:
  internet:
    sst: 1
    sd: 0xFFFFFF # explicitly set sd to avoid core regection
"""
)

In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    project: post5g-beta
    experiment: 5g-zmq
    flavor: xlarge # ocudu zmq needs more than 5 CPU cores (see: https://github.com/srsran/srsRAN_Project/issues/1472)
    extra_path: "" # Use SLICES CLI from PATH

ran:
  implementation: ocudu
  components:
    gnb:
      type: gnb
      advanced_config:
        inactivity_timer: 7200
        srate: 11.52
        base_srate: "11.52e6"
        pdcch:
          common:
            ss0_index: 0
            coreset0_index: 6
          dedicated:
            ss2_type: common
            dci_format_0_1_and_1_1: false
        pdsch:
          mcs_table: qam64
        pusch:
          mcs_table: qam64

ues:
  sim_ue:
    type: containerized
    implementation: srsue
    advanced_config:
      srate: "11.52e6"
"""
)

## Provisioning:

In [ ]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await prov.provision()

## Kubernetes Deployment:

In [ ]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(prov)

In [ ]:
await k8s.deploy()

## 5G Core Deployment:

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(k8s)

In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()


In [ ]:
# Preserve the database and subscriber steps performed by upstream core.deploy().
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


## RAN Deployment:

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(core)

from nador import MonitoringDeployment, configure_ran_kpis
ran_kpi_config = MonitoringDeployment.from_yaml_file("monitoring.yaml")
configure_ran_kpis(ran, ran_kpi_config)

In [ ]:
configure_ran_kpis(ran, ran_kpi_config)
await ran.deploy()

Get OCUDU ZMQ gNB logs:

In [ ]:
print(await ran.get_logs(quiet=True))

## UE Deployement:

## Monitoring before UE attachment

The profile uses the control host selected by Kubernetes, including physical
Sopnodes. Capture interfaces refer to pod interfaces, not the host or radio NIC.
Keep the existing provisioning, RF and Multus settings. Grafana uses `admin` / `admin`.

In [ ]:
from pathlib import Path
import asyncio
import json

from nador import MonitoringManagement
from nador.models import MonitoringDeployment

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)


In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )

### Open Grafana

The SSH tunnel uses the current model address and jump host.
Open the link after deployment; close the tunnel with `grafana_tunnel.terminate()`.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))

In [ ]:
from orchestrator.deployment.ue import UEManagement

ue = UEManagement(preceding=ran)

In [ ]:
await ue.deploy()

Get SRSUE ZMQ UE logs:

In [ ]:
print(await ue.get_logs(quiet=True))

Get gNB logs after UE was connected:

In [ ]:
print(await ran.get_logs(quiet=True))

Veify that the UE got a PDU session:

In [ ]:
print((await ue.run_command(command=["ip", "addr"], quiet=True, flatten=False))["sim_ue"])


#### Test connectivity:

In [ ]:
print((await ue.run_command(command=["apt", "update"], quiet=True, flatten=False))["sim_ue"])


In [ ]:
print((await ue.run_command(command=["apt", "install", "-y", "iputils-ping"], quiet=True, flatten=False))["sim_ue"])


In [ ]:
print(
    (await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "tun_srsue"], quiet=True, flatten=False))[
        "sim_ue"
    ]
)


In [ ]:
print(
    (await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "tun_srsue"], quiet=True, flatten=False))[
        "sim_ue"
    ]
)


## Iperf with experiment artifacts

The server runs on the Kubernetes control host from the deployment model.
Each client binds to its own 5G interface. Review the explicit scenario before running it.
The collector exports Prometheus data after traffic and saves each report with the timeline.

In [ ]:
from datetime import UTC, datetime
from nador.artifacts import ExperimentRun
from orchestrator.deployment import ProvisioningManagement
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = monitoring.deployment_model.params.k8s.master_node
CLIENTS = ["sim_ue"]
UE_INTERFACE = "tun_srsue"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5

server_node = monitoring.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname or SERVER_NODE
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(monitoring.deployment_model.params.ansible.runner_path) / "iperf-server")
clients = ue

await run_ansible_playbook(
    playbook=[
        {
            "name": "Install iperf3 on the server host",
            "hosts": SERVER_NODE,
            "gather_facts": False,
            "become": True,
            "tasks": [
                {
                    "ansible.builtin.apt": {
                        "name": "iperf3",
                        "state": "present",
                        "update_cache": True,
                        "cache_valid_time": 3600,
                    }
                }
            ],
        }
    ],
    inventory=server_inventory,
    runner_path=server_runner,
    quiet=True,
)
await clients.run_command(command=["apt-get", "update"], components=CLIENTS, quiet=True, flatten=False)
await clients.run_command(
    command=["env", "DEBIAN_FRONTEND=noninteractive", "apt-get", "install", "-y", "iperf3"],
    components=CLIENTS,
    quiet=True,
    flatten=False,
)
addresses = await clients.run_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE], components=CLIENTS, quiet=True, flatten=False
)
ue_ips = {}
for client in CLIENTS:
    ipv4 = [
        a["local"] for device in json.loads(addresses[client]) for a in device["addr_info"] if a["family"] == "inet"
    ]
    if len(ipv4) != 1:
        raise RuntimeError(f"{client}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[client] = ipv4[0]
print(f"Server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")


In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60", "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE,
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await clients.run_command(
        command=[
            "iperf3",
            "-c",
            server_ip,
            "-B",
            ue_ips[client],
            "--bind-dev",
            UE_INTERFACE,
            "-p",
            str(port),
            "-t",
            str(duration),
            "--connect-timeout",
            "10000",
            "-J",
            "--get-server-output",
            *reverse,
        ],
        components=[client],
        quiet=True,
        flatten=False,
    )
    report = json.loads(output[client])
    if report.get("error"):
        raise RuntimeError(report["error"])
    return report


In [ ]:
SCENARIO = [
    {"name": "ue1_uplink", "client": 'sim_ue', "direction": "uplink", "duration": DURATION},
    {"name": "ue1_downlink", "client": 'sim_ue', "direction": "downlink", "duration": DURATION},
]

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-host-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {test['name']}: {test['client']}, {test['direction']}, {test['duration']} seconds")
        await start_iperf_server(port)

        with experiment.section(test["name"]):
            report = await iperf_client(
                client=test["client"],
                direction=test["direction"],
                duration=test["duration"],
                port=port,
            )

        experiment.save_json(f"{test['name']}.json", report)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")

In [ ]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

# Test OCUDU ZMQ gNB with OAI NR UE:

Reference: https://docs.ocudu.org/tutorials/oaiue/#zeromq-based-setup

#### Add a second OCUDU gNB:

In [ ]:
metamodel, deployment_model = ran.output()

In [ ]:
from metamodel.models.ru import RU

metamodel.rus["sim_ru_gnb2"] = RU(txGainDB=-12, rxGainDB=0)

In [ ]:
ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
from metamodel.models.ran import GNBComponent
from orchestrator.models.ran import DeployedRANComponent

ran.add_component(
    name="gnb2",
    component_meta=GNBComponent.from_yaml(
"""
type: gnb
config:
  nrCellId: 2
  physicalCellId: 1
  ru: sim_ru_gnb2
  radioConfiguration:
    nrBand: 78
    downlinkArfcn: 632628
    ssbArfcn: 632256
    pointAFrequencyArfcn: 632016
    channelBandwidthMhz: 20
    prachConfigurationIndex: 159
    subcarrierSpacingKhz: 30
    ssbPeriodicityMs: 10
    tddConfiguration:
      periodicityMilliseconds: 2.5
      nrofDownlinkSlots: 3
      nrofDownlinkSymbols: 10
      nrofUplinkSlots: 1
      nrofUplinkSymbols: 2
"""
    ),
    component_deployment=DeployedRANComponent.from_yaml(
"""
type: gnb
implementation: ocudu
advanced_config:
  srate: 23.04
  csi:
    csi_rs_enabled: true
  pucch:
    nof_cell_csi_res: 1
"""
    ),
)

Deploy `gnb2`:

In [ ]:
configure_ran_kpis(ran, ran_kpi_config)
await ran.deploy(components=['gnb2'])

Check its logs:

In [ ]:
print(await ran.get_logs(components=["gnb2"], quiet=True))

Add the OAI NR UE to be connected to OCUDU `gnb2` over ZeroMQ:

In [ ]:
ue = UEManagement(ran)

In [ ]:
from metamodel.models.ue import UE
from orchestrator.models.ue import DeployedUE

ue.add_component(
    name="sim_ue2",
    component_meta=UE.from_yaml(
"""
config:
  supi: "001010000000101"
  ru: sim_ru_gnb
  gnbs:
    - gnb2
"""
),
    component_deployment=DeployedUE.from_yaml(
"""
type: containerized
implementation: oai
"""
),
)

Add it to the 5G core database:

In [ ]:
core = CoreManagement(ue)
await core.register_ues(ues=["sim_ue2"])

Deploy it:

In [ ]:
await ue.deploy(components=["sim_ue2"])

Check it's logs:

In [ ]:
print(await ue.get_logs(quiet=True))

Check if it got a PDU session:

In [ ]:
print((await ue.run_command(command=["ip", "addr"], quiet=True, flatten=False))["sim_ue2"])


Check connectivity:

### Verify monitoring for both gNBs

Run after deploying the updated monitoring controller. These cells check that Prometheus can scrape both gNB probes, then record UE2 latency counters before the existing ping tests.

In [ ]:
from urllib.parse import urlencode
from urllib.request import urlopen

async def monitoring_query(url, expression):
    def fetch():
        with urlopen(url + "/api/v1/query?" + urlencode({"query": expression}), timeout=10) as response:
            payload = json.load(response)
        if payload["status"] != "success":
            raise RuntimeError(payload)
        return payload["data"]["result"]
    return await asyncio.to_thread(fetch)

ue2_imsi = str(ue.metamodel.ues["sim_ue2"].config.supi)
gnb2_prefix = f"{ue.deployment_model.ran.implementation}-gnb2-"
ue2_latency_query = (
    'sum by (pod, probe_role, ue_ip, slice) '
    f'(gtp_teid_latency_observed_ns_count{{imsi="{ue2_imsi}"}})'
)

In [ ]:
from nador.artifacts import prometheus_connection

async with prometheus_connection(monitoring) as prometheus_url:
    probes = await monitoring_query(prometheus_url, 'up{job="latency-probes"}')
    for component in ("gnb", "gnb2"):
        prefix = f"{ue.deployment_model.ran.implementation}-{component}-"
        matches = [row for row in probes if row["metric"].get("pod", "").startswith(prefix)]
        assert matches and all(float(row["value"][1]) == 1 for row in matches), f"{component}: no healthy probe"
        print(f"{component}: Prometheus probe is UP")

    baseline = await monitoring_query(prometheus_url, ue2_latency_query)

ue2_latency_before = {
    (row["metric"]["pod"], row["metric"]["probe_role"]): float(row["value"][1])
    for row in baseline
}
print(f"Now run the UE2 ping cells below (IMSI {ue2_imsi}).")

### Wait for UE2 attachment

A ready pod only means the UE process has started. OAI creates `oaitun_ue1`
and assigns its IPv4 address after attachment and PDU-session establishment.
Wait up to 120 seconds before testing connectivity. If the tunnel never receives
an IPv4 address, stop here and inspect UE2/gNB2 logs; waiting does not repair a failed attachment.


In [ ]:
import shlex

await ue.run_command(
    command=[
        "sh", "-c",
        shlex.quote("""for attempt in $(seq 1 60); do
    if ip -4 -o addr show dev oaitun_ue1 2>/dev/null | grep -q ' inet '; then
        exit 0
    fi
    sleep 2
done
echo 'UE2 did not receive an IPv4 address on oaitun_ue1 within 120 seconds' >&2
exit 1
"""),
    ],
    components=["sim_ue2"],
    quiet=True,
)
print("UE2 tunnel has an IPv4 address; ready for connectivity tests.")


In [ ]:
print(
    (
        await ue.run_command(
            command=["ping", "11.1.1.1", "-c", "6", "-I", "oaitun_ue1"],
            components=["sim_ue2"],
            quiet=True,
            flatten=False,
        )
    )["sim_ue2"]
)


In [ ]:
print(
    (
        await ue.run_command(
            command=["ping", "8.8.8.8", "-c", "6", "-I", "oaitun_ue1"],
            components=["sim_ue2"],
            quiet=True,
            flatten=False,
        )
    )["sim_ue2"]
)


### Check UE2 mapping and new latency observations

Run after the two UE2 pings above. Both the second gNB and UPF must have new observations labelled with UE2’s IMSI, IP and slice.

In [ ]:
from nador.artifacts import prometheus_connection

async with prometheus_connection(monitoring) as prometheus_url:
    observations = await monitoring_query(prometheus_url, ue2_latency_query)

observed_roles = set()
for row in observations:
    labels = row["metric"]
    pod, role = labels["pod"], labels["probe_role"]
    if role == "gnb" and not pod.startswith(gnb2_prefix):
        continue
    added = float(row["value"][1]) - ue2_latency_before.get((pod, role), 0)
    if added <= 0:
        continue
    assert labels.get("ue_ip") and labels.get("slice") not in (None, "", "unknown"), "UE2 mapping is incomplete"
    observed_roles.add(role)
    print(f"{pod}: UE {ue2_imsi}, IP {labels['ue_ip']}, slice {labels['slice']}; +{added:g} latency observations")

assert {"gnb", "upf"} <= observed_roles, "UE2 needs fresh latency observations at both gnb2 and UPF"
print("PASS: both gNB probes are up, and UE2 has mapped latency observations at gnb2 and UPF.")

In [ ]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")